# KS Test Drift Detection

This notebook detects data drift by comparing feature distributions across time periods. It tests **all numeric model inputs** (KS test) and **categorical features** (chi-square test on category frequencies) between the **training and testing splits** (chronological 64% train / 20% test, consistent with the ML pipeline) using data from `mle.gold.product_city_demand`. Results are logged to `mle.monitoring.model_drift_ks_results` with per-feature `test_method` (`ks` / `chi2`) and a `drift_detected` flag. Consecutive-month comparisons are also logged for dashboard continuity. Leakage columns (`ordered_quantity`, `total_cost_usd`) are excluded since they are not model inputs.

---

## 1. Load Data from Gold Table

Load the demand data from the Unity Catalog `mle.gold.product_city_demand` table for analysis.


In [0]:
# Load data
from datetime import datetime
from pyspark.sql.functions import col, month, year
import pandas as pd

# Features to monitor for drift (model inputs + target; leakage columns excluded)
NUMERIC_FEATURES = [
    "demand_quantity",                # target
    "avg_available_inventory",        # strongest non-leakage predictor (r=0.64)
    "avg_product_cost_usd",
    "avg_unit_price_usd",
    "avg_supplier_reliability_score",
    "transaction_count",             # volume signal
]
CATEGORICAL_FEATURES = ["product_name", "product_category", "destination_city"]
# ordered_quantity / total_cost_usd excluded: post-event leakage columns, not model inputs

# Load table
spark_df = spark.table("mle.gold.product_city_demand")
spark_df = spark_df.withColumn('transaction_month', month(col('transaction_date')))
spark_df = spark_df.withColumn('transaction_year', year(col('transaction_date')))
cols = ['transaction_date', 'transaction_month', 'transaction_year'] + NUMERIC_FEATURES + CATEGORICAL_FEATURES
pd_df = spark_df.select(*cols).toPandas()
pd_df['transaction_date'] = pd.to_datetime(pd_df['transaction_date'])

pd_df.head()

## 2. Drift Tests: Training vs Testing Distributions (all features)

This step splits the data chronologically (64% train / 16% val / 20% test, the same split used by the ML pipeline) and tests every monitored feature between the training and testing periods: **KS test** for numeric features (`demand_quantity`, `avg_available_inventory`, `avg_product_cost_usd`, `avg_unit_price_usd`, `avg_supplier_reliability_score`, `transaction_count`) and **chi-square test** for categorical features (`product_name`, `product_category`, `destination_city` — detects shifts in the product/city mix). Each feature is also tested across the last two consecutive months for dashboard continuity. Drift is flagged when p-value < 0.05. Results are logged per feature with `test_method` and `drift_detected` columns.

In [0]:
# Compute drift: KS tests (numeric) + chi-square tests (categorical), train vs test + consecutive months
from scipy.stats import ks_2samp, chi2_contingency
import numpy as np
import pyspark.sql.types as T

ALPHA = 0.05
results = []

# --- Chronological split (64% train / 16% val / 20% test) — same approach as ml_pipeline ---
dates = pd.DatetimeIndex(sorted(pd_df['transaction_date'].unique()))
n_dates = len(dates)
if n_dates < 5:
    raise ValueError("Not enough distinct transaction dates to build train/test splits.")
n_train = int(n_dates * 0.64)
n_val = int(n_dates * 0.16)
train_end, test_start = dates[n_train - 1], dates[n_train + n_val]

train_df = pd_df[pd_df['transaction_date'] <= train_end]
test_df = pd_df[pd_df['transaction_date'] >= test_start]
print(f"Train: {len(train_df)} rows ({dates[0].date()} to {train_end.date()})")
print(f"Test:  {len(test_df)} rows ({test_start.date()} to {dates[-1].date()})")

# --- Consecutive-month slices for dashboard continuity ---
pairs = sorted(set(zip(pd_df['transaction_year'], pd_df['transaction_month'])))
(current_year, current_month) = pairs[-1]
(previous_year, previous_month) = pairs[-2]
prev_df = pd_df[(pd_df['transaction_year'] == previous_year) & (pd_df['transaction_month'] == previous_month)]
cur_df = pd_df[(pd_df['transaction_year'] == current_year) & (pd_df['transaction_month'] == current_month)]

def log_result(feature, stat, p, method, comparison_type, ref_month, ref_year, cur_month, cur_year, ref_rows, cur_rows):
    results.append({
        "eval_date": str(datetime.now().date()),
        "feature": feature,
        "reference_month": int(ref_month),
        "reference_year": int(ref_year),
        "current_month": int(cur_month),
        "current_year": int(cur_year),
        "ks_statistic": float(stat),
        "p_value": float(p),
        "comparison_type": comparison_type,
        "train_rows": int(ref_rows) if ref_rows is not None else None,
        "test_rows": int(cur_rows) if cur_rows is not None else None,
        "test_method": method,
        "drift_detected": bool(p < ALPHA),
    })

def drift_label(p):
    return "DRIFT" if p < ALPHA else "no drift"

# --- Numeric features: KS test ---
print(f"\n=== Numeric features: KS test (alpha = {ALPHA}) ===")
for feat in NUMERIC_FEATURES:
    a = train_df[feat].dropna()
    b = test_df[feat].dropna()
    ks_stat, p_value = ks_2samp(a, b)
    print(f"  {feat:<30} train vs test: KS={ks_stat:.4f}, p={p_value:.4f} ({drift_label(p_value)}) | mean {a.mean():.2f} -> {b.mean():.2f}")
    log_result(feat, ks_stat, p_value, "ks", "train_vs_test",
               train_end.month, train_end.year, dates[-1].month, dates[-1].year, len(a), len(b))

    a2 = prev_df[feat].dropna()
    b2 = cur_df[feat].dropna()
    m_ks, m_p = ks_2samp(a2, b2)
    print(f"  {feat:<30} {previous_year}-{previous_month:02d} vs {current_year}-{current_month:02d}: KS={m_ks:.4f}, p={m_p:.4f} ({drift_label(m_p)})")
    log_result(feat, m_ks, m_p, "ks", "consecutive_months",
               previous_month, previous_year, current_month, current_year, len(a2), len(b2))

# --- Categorical features: chi-square test on category frequencies ---
def chi2_test(col_a, col_b):
    """Chi-square test comparing category frequency distributions of two samples."""
    cats = sorted(set(col_a.dropna().unique()) | set(col_b.dropna().unique()))
    if not cats:
        return None, None
    counts_a = col_a.value_counts()
    counts_b = col_b.value_counts()
    table = np.array([[int(counts_a.get(c, 0)) for c in cats],
                      [int(counts_b.get(c, 0)) for c in cats]])
    if (table.sum(axis=1) == 0).any() or table.sum() == 0:
        return None, None
    chi2, p, _, _ = chi2_contingency(table)
    return chi2, p

print(f"\n=== Categorical features: chi-square test (alpha = {ALPHA}) ===")
for feat in CATEGORICAL_FEATURES:
    chi2, p = chi2_test(train_df[feat], test_df[feat])
    if chi2 is not None:
        print(f"  {feat:<30} train vs test: chi2={chi2:.4f}, p={p:.4f} ({drift_label(p)})")
        log_result(feat, chi2, p, "chi2", "train_vs_test",
                   train_end.month, train_end.year, dates[-1].month, dates[-1].year,
                   len(train_df), len(test_df))

    chi2_m, p_m = chi2_test(prev_df[feat], cur_df[feat])
    if chi2_m is not None:
        print(f"  {feat:<30} {previous_year}-{previous_month:02d} vs {current_year}-{current_month:02d}: chi2={chi2_m:.4f}, p={p_m:.4f} ({drift_label(p_m)})")
        log_result(feat, chi2_m, p_m, "chi2", "consecutive_months",
                   previous_month, previous_year, current_month, current_year,
                   len(prev_df), len(cur_df))

n_drift = sum(1 for r in results if r["drift_detected"])
print(f"\nSummary: {len(results)} tests run, {n_drift} with significant drift (p < {ALPHA})")

# --- Persist to monitoring table (schema evolves: adds test_method + drift_detected) ---
schema = T.StructType([
    T.StructField("eval_date", T.StringType()),
    T.StructField("feature", T.StringType()),
    T.StructField("reference_month", T.IntegerType()),
    T.StructField("reference_year", T.IntegerType()),
    T.StructField("current_month", T.IntegerType()),
    T.StructField("current_year", T.IntegerType()),
    T.StructField("ks_statistic", T.DoubleType()),
    T.StructField("p_value", T.DoubleType()),
    T.StructField("comparison_type", T.StringType()),
    T.StructField("train_rows", T.IntegerType()),
    T.StructField("test_rows", T.IntegerType()),
    T.StructField("test_method", T.StringType()),
    T.StructField("drift_detected", T.BooleanType()),
])

spark.sql("CREATE SCHEMA IF NOT EXISTS mle.monitoring")
spark_df_out = spark.createDataFrame(results, schema)
spark_df_out.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("mle.monitoring.model_drift_ks_results")
print(f"\n{len(results)} drift results saved to mle.monitoring.model_drift_ks_results")
print("  comparison_type = train_vs_test / consecutive_months")
print("  test_method = ks (numeric) / chi2 (categorical), drift_detected = p < 0.05")